# Basic Component

### LLMS

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")  # or "gpt-3.5-turbo"
response = llm.invoke("Write a haiku about autumn leaves")
print(response.content)


Crisp whispers descend,  
Crimson and gold dance the ground,  
Nature's last bow call.


## prompt templates

In [3]:
from langchain.prompts import PromptTemplate

template = "Translate the following text into French: {text}"
prompt = PromptTemplate.from_template(template)

formatted = prompt.format(text="Hello, how are you?")
print(formatted)
llm.invoke(formatted)



Translate the following text into French: Hello, how are you?


AIMessage(content='Bonjour, comment ça va ?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 20, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CGjSfTVZWytRCGaLXLrIpTTDJLG3D', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--94412b1d-3332-4e11-9b9b-bf46cef957c6-0', usage_metadata={'input_tokens': 20, 'output_tokens': 6, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## chains

In [7]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=prompt)
print(chain.invoke({"text": "I love programming"}))


{'text': "J'aime programmer."}


### Memory conversation

In [10]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory)

print(conversation.invoke("Hi, my name is Alex"))
print(conversation.invoke("What's my name?"))  # remembers context


{'input': 'Hi, my name is Alex', 'history': '', 'response': "Hello, Alex! It's great to meet you. I'm here to chat about anything you'd like—whether it's questions, ideas, or just some friendly banter. What’s on your mind today?"}
{'input': "What's my name?", 'history': "Human: Hi, my name is Alex\nAI: Hello, Alex! It's great to meet you. I'm here to chat about anything you'd like—whether it's questions, ideas, or just some friendly banter. What’s on your mind today?", 'response': "Your name is Alex! It's nice to meet you, Alex. Is there anything specific you'd like to talk about today?"}


### Retrievers (Docs + Vector DB)

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Step 1: Split documents
docs = ["LangChain is a framework for LLM apps", "LlamaIndex is great for retrieval"]
splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=10)
chunks = splitter.create_documents(docs)

# Step 2: Store in vector DB
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

# Step 3: Retrieval
retriever = vectorstore.as_retriever()
results = retriever.invoke("What is LangChain?")
print(results[0].page_content)
print(results)


LangChain is a framework for
[Document(id='27e74a75-4447-488e-8cc8-0bfd039d4cab', metadata={}, page_content='LangChain is a framework for'), Document(id='d4cc4218-b36f-4ae0-9111-1b6bce4faa5c', metadata={}, page_content='for LLM apps'), Document(id='7a5e675b-05cb-4228-9d43-932a9a0fc32a', metadata={}, page_content='LlamaIndex is great for'), Document(id='bd3dd8ae-ca56-419f-b05e-680df8ebb496', metadata={}, page_content='great for retrieval')]


### Agents (LLM + Tools)

In [9]:
from langchain.agents import initialize_agent, load_tools

tools = load_tools(["llm-math"], llm=llm)
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)

agent.invoke("What is 13.4 * 9.6?")


/tmp/ipykernel_7374/697595961.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)




> Entering new AgentExecutor chain...
To calculate \( 13.4 \times 9.6 \), I will use a calculator. 

Action: Calculator
Action Input: 13.4 * 9.6
Observation: Answer: 128.64
Thought:I now know the final answer.  
Final Answer: 128.64

> Finished chain.


{'input': 'What is 13.4 * 9.6?', 'output': '128.64'}